# ED tokenizers

Declare the alphabet, import the tokenizers, and train them, as `train_tokenizers.ipynb` does for OnHW. Every vocabulary is learned from **IAM training data only**: linguistic tokenizers from the transcriptions, handwriting tokenizers from the connectivity evidence exported by DTLR. ED only supplies its declared alphabet. Outputs are shared across WD/WI folds. Run from the TVA root with the WSL tva kernel.

In [ ]:
from pathlib import Path
from tva.tokenizers import CharacterTokenizer, get_tokenizer
from tva.handwriting_tokenizers import HandwritingBigramTokenizer
from tva.handwriting_tokenizers import HandwritingBPETokenizer, HandwritingUnigramTokenizer
from tva.handwriting_tokenizers import UnigramEMTokenizer
from tva.compression_matched_tokenizers import build_fold_tokenizers
from tva.dataset.iam import prepare_iam
from tva.handwriting.common import save_frozen

# Run from the TVA repository root with the WSL tva kernel.
iam_labels = Path('/home/artellisys/DTLR/data/IAM_new/labels.pkl')
iam_selection = Path('/home/artellisys/dtlr-output/iam-train-full/selection.json')
iam_evidence = Path('artifacts/evidence/iam/iam_connectivity_v1.json.gz')
iam_dir = 'data/tva/iam'
output = Path('artifacts/tokenizers/ed')

categories = [
    '', ' ', '!', '"', '%', "'", '(', ')', '+', ',', '-', '.', '/',
    '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '=', '?',
    'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M',
    'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z',
    'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm',
    'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z',
]

## IAM in TVA format

Write the 5,694 IAM training transcriptions as a one-fold TVA dataset (`train.json`). The bigram tokenizers below train on this directory exactly like TVA trains on an OnHW dataset.

In [ ]:
prepare_iam(iam_labels, iam_selection, iam_dir)

## Character tokenizer

The character baseline uses the unchanged original TVA class and loads its standard 79-class JSON file. The file was generated with TVA’s own CharacterTokenizer.train() from the declared alphabet; no vocabulary selection from labels is involved. All five fold files contain the same vocabulary.


In [ ]:
tokenizer_char = CharacterTokenizer()
tokenizer_char.load(output / 'character' / 'char' / '0.json')
print('Character classes:', tokenizer_char.size)


## Linguistic bigram — original TVA

The unchanged TVA `BigramTokenizer`: the most frequent adjacent pairs in the unique IAM labels, greedy encoding. TVA breaks frequency ties in `set()` order, so start Jupyter with `PYTHONHASHSEED=42` to reproduce the frozen file. `save_frozen` refuses to overwrite a different existing file.

In [ ]:
tokenizer_bigram = get_tokenizer('bigram')
tokenizer_bigram.train(iam_dir, categories, 224)
save_frozen(output / 'linguistic' / 'bigram' / '0.json',
            Path(iam_dir, 'tokenizers', 'bigram224', '0.json').read_bytes())
print('Linguistic classes:', tokenizer_bigram.size)

## Linguistic bigram — letters only

The same TVA trainer with `letters_only=True`: pairs containing spaces, digits or punctuation are not counted, so all 145 bigrams are letter pairs, like the handwriting bigram. Spaces and punctuation stay in the categories as single-character tokens. Only frequency versus connectivity then separates the two vocabularies.

In [ ]:
tokenizer_bigram_letters = get_tokenizer('bigram')
tokenizer_bigram_letters.train(iam_dir, categories, 224, letters_only=True)
save_frozen(output / 'linguistic' / 'bigram_letters' / '0.json',
            Path(iam_dir, 'tokenizers', 'bigram224_letters', '0.json').read_bytes())
print('Letters-only linguistic classes:', tokenizer_bigram_letters.size)

## Linguistic BPE — original TVA, letters only

TVA's `BPETokenizer` with `letters_only=True`: merges only join letters, and spaces, digits and punctuation stay single tokens (TVA's default whitespace splitting would drop the spaces of ED sentences). 145 merges on top of the blank and the 78 characters.

In [ ]:
tokenizer_linguistic_bpe = get_tokenizer('bpe')
tokenizer_linguistic_bpe.train(iam_dir, categories, 224, letters_only=True)
save_frozen(output / 'linguistic' / 'bpe' / '0.json',
            Path(iam_dir, 'tokenizers', 'bpe224_letters', '0.json').read_bytes())
print('Linguistic BPE classes:', tokenizer_linguistic_bpe.size)

## Linguistic Unigram — original TVA, letters only

TVA's `UnigramTokenizer` with `letters_only=True`. Hugging Face's Unigram trainer is not bit-reproducible: retraining selects the same 145 pieces but in a different order with slightly different scores, so `save_frozen` refuses to overwrite the frozen file. Delete it deliberately to refit.

In [ ]:
tokenizer_linguistic_unigram = get_tokenizer('unigram')
tokenizer_linguistic_unigram.train(iam_dir, categories, 224, letters_only=True)
save_frozen(output / 'linguistic' / 'unigram' / '0.json',
            Path(iam_dir, 'tokenizers', 'unigram224_letters', '0.json').read_bytes())
print('Linguistic Unigram classes:', tokenizer_linguistic_unigram.size)

## Linguistic Unigram — same EM as the handwriting Unigram

The linguistic partner of the handwriting Unigram below. It runs the same EM and pruning on the same IAM lines, but every letter span of 2–6 characters seen at least 20 times is a candidate and starts from its frequency instead of its connected count. Only the connectivity evidence then separates the two vocabularies. TVA's Unigram above stays as the reference baseline. Training is deterministic.

In [ ]:
tokenizer_unigram_em = UnigramEMTokenizer()
tokenizer_unigram_em.train(iam_dir, categories, 224)
save_frozen(output / 'linguistic' / 'unigram_em' / '0.json',
            Path(iam_dir, 'tokenizers', 'unigram_em224', '0.json').read_bytes())
print('Linguistic Unigram (EM) classes:', tokenizer_unigram_em.size)

## Handwriting bigram — most connected IAM bigrams

Same class structure, files and encoding as TVA's `BigramTokenizer`; only the selection differs. Among letter pairs with at least 20 exactly aligned IAM observations, it keeps the 145 with the highest connected rate (how often the two letters are written connected). `policy='greedy'` encodes exactly like the linguistic bigram; `policy='utility-based'` instead maximizes the total connected rate. No ED labels are read.

In [ ]:
tokenizer_handwriting = HandwritingBigramTokenizer(policy='greedy')
tokenizer_handwriting.train(iam_dir, categories, 224, path_evidence=iam_evidence)
save_frozen(output / 'handwriting' / 'bigram' / '0.json',
            Path(iam_dir, 'tokenizers', 'handwriting_bigram224', '0.json').read_bytes())
print('Handwriting classes:', tokenizer_handwriting.size)

## Handwriting BPE — merges ranked by connectivity

Subclass of TVA's `BPETokenizer`; only `.train()` differs. Candidate pieces are letter spans of 2–6 characters with at least 20 aligned IAM observations, written connected at least half the time. Like BPE training, it repeatedly merges the most supported pair, but support counts only connected occurrences. Encoding is TVA's (Hugging Face) BPE with the same letters-only splitting as the linguistic BPE. No ED labels are read.

In [ ]:
tokenizer_bpe = HandwritingBPETokenizer()
tokenizer_bpe.train(iam_dir, categories, 224, path_evidence=iam_evidence)
save_frozen(output / 'handwriting' / 'bpe' / '0.json',
            Path(iam_dir, 'tokenizers', 'handwriting_bpe224', '0.json').read_bytes())
print('Handwriting BPE classes:', tokenizer_bpe.size)

## Handwriting Unigram — pieces seeded by connectivity

Subclass of TVA's `UnigramTokenizer`; only `.train()` differs. The same candidates start from their connected counts; EM on the IAM text and likelihood-based pruning reduce them to 145 pieces. Encoding is TVA's (Hugging Face) Unigram Viterbi with letters-only splitting. Unlike TVA's Unigram, this training is deterministic. No ED labels are read.

In [ ]:
tokenizer_unigram = HandwritingUnigramTokenizer()
tokenizer_unigram.train(iam_dir, categories, 224, path_evidence=iam_evidence)
save_frozen(output / 'handwriting' / 'unigram' / '0.json',
            Path(iam_dir, 'tokenizers', 'handwriting_unigram224', '0.json').read_bytes())
print('Handwriting Unigram classes:', tokenizer_unigram.size)

## Compression-matched text-only controls

Keep the top k bigrams of the letters-only linguistic bigram or the first k linguistic BPE merges, with k chosen so that IAM text is compressed as much as by the handwriting partner. A Unigram has no learned order, so the Unigram control retrains the text-only Unigram EM (`UnigramEMTokenizer`, above) with a budget of k pieces instead. The bigram control loads with `BigramTokenizer`, the BPE control with `BPETokenizer` and the Unigram control with `UnigramTokenizer`; `selection.json` records the choice of k. Keeping the top k TVA bigrams is exactly `BigramTokenizer.train(..., 1 + 78 + k, letters_only=True)`, because TVA ranks bigrams by frequency; it needs the letters-only bigram trained above.

In [ ]:
# 'unigram' retrains the EM for every budget k (several minutes).
for kind in ('bigram', 'bpe', 'unigram'):
    print('Saved', build_fold_tokenizers(kind))

## Encode and decode


In [ ]:
text = 'In short, there will be on TV!'
for tokenizer in (tokenizer_char, tokenizer_bigram, tokenizer_bigram_letters, tokenizer_linguistic_bpe, tokenizer_linguistic_unigram, tokenizer_unigram_em, tokenizer_handwriting, tokenizer_bpe, tokenizer_unigram):
    ids = tokenizer.encode(text)
    print(type(tokenizer).__name__, ids, tokenizer.decode(ids))

Every learned tokenizer has **224 classes** (blank, 78 characters, 145 learned pieces); the compression-matched controls are smaller. Their files already exist, so recognition training can use them immediately. To use an existing tokenizer without fitting, call `.load(path)`.

Detailed checks live outside this notebook:
~~~bash
python -m tva.handwriting.audit --compare --dataset-directory /mnt/c/Users/Ali/Downloads/fau-english-dataset
~~~

Recognition-model training remains separate and manual, using configs/thesis/ed/ (one folder per experiment, see its README).

## Save fold files

Publish `0.json` through `4.json` in each tokenizer directory, matching the original TVA layout. These are identical copies of the tokenizer fitted above from IAM; no ED fold labels are used. Recognition configs point to the directory and load the requested fold.


In [ ]:
from tva.handwriting.common import save_fold_tokenizers

for family, methods in {'character': ['char'], 'linguistic': ['bigram', 'bigram_letters', 'bpe', 'unigram', 'unigram_em'], 'handwriting': ['bigram', 'bpe', 'unigram']}.items():
    for method in methods:
        save_fold_tokenizers(output / family / method)